Phase I Project Proposal

Spurious Stock Sleuth: Investigating strange correlations in the market

Name: Santiago Annunziato, DS 3000


### Introduction

“I can calculate the motion of heavenly bodies, but not the madness of people.”-Isaac Newton

Extrapolating from the words of Isaac Newton, while the financial market is comprised and influenced by innumerable 
variables and factors, such as interest rates, GDP, and other traditional economic indicators,there is the question
of whether more unconventional factors may also be in part, at play, and potentially even related.
From the 2021 GameStop Stock fiasco, to the 2025 Sydney Sweeny American Eagle Stock surge, we can observe how
non standard influences not inherently tied to traditional market trends, can have a significant impact on the market.
In recent years, the proliferation data driven market analysis begs the question, of how many unforseen patterns may
exist in actuality, seemingly inexplicable patterns, which further begs the question of whether such an apparent observed
correlation, is merely spurious, or potentially sufficently predictive?

Spurious correlations—statistical relationships between variables that appear connected but are actually driven by 
coincidence or hidden confounding factors—pose a significant risk in financial analysis. Tyler Vigen's 
work on spurious correlations has popularized examples ranging from the humorous (margarine consumption vs. divorce rates in Maine) 
to the thought-provoking (Nicholas Cage films vs. swimming pool drownings). While entertaining, these examples 
underscore a serious methodological concern: in the age of data mining, the ease of finding correlations has far 
outpaced our ability to verify causation.

This project investigates spurious correlations in the stock market by examining three major publicly traded 
companies across different sectors: JPMorgan Chase (JPM) in finance, Tesla (TSLA) in technology, and Target 
(TGT) in retail. By systematically exploring correlations between these stock prices and seemingly unrelated 
variables—such as demographic trends, internet search behavior, and various other cultural phenomena—this 
study aims to accomplish two goals. First, to identify which unconventional variables exhibit strong statistical 
correlations with stock prices. Second, to investigate the underlying mechanisms behind these correlations, 
distinguishing between genuine (if indirect) causal relationships and purely coincidental patterns.

The practical implications of this research extend to both individual investors and financial institutions. 
Understanding the difference between meaningful indicators and spurious correlations can prevent costly investment 
decisions based on false patterns. Moreover, by identifying the confounding variables that create apparent 
relationships, this analysis contributes to a more sophisticated understanding of market dynamics and the complex 
web of economic factors that truly influence stock prices.

### RESEARCH QUESTIONS:

QUERY 1: Do spurious correlations exist between chosen stocks, and extraneous seemingly unrelated trends?

QUERY 2: Are this spurious correlations purely coincidental, or indicative of an unknown confounding variable AND
        #are the correations sufficient to be predictive?

### Data Collection

This project utilizes publicly available data from multiple sources, accessed programmatically through 
Python APIs.

#### Stock Price Data

Stock price data for JPMorgan Chase (JPM), Tesla (TSLA), and Target (TGT) is collected using the yfinance 
Python library, which provides free access to Yahoo Finance data. The analysis spans from 2010 to 2022, 
chosen because Tesla's IPO occurred in June 2010. This 12-year period provides over 3,000 trading days of 
data for each stock and captures significant market events including the post-2008 financial crisis recovery, 
the tech boom of the 2010s, and the COVID-19 pandemic market disruption.

In [63]:
!pip install yfinance
!pip install pytrends

In [64]:
import yfinance as yf
import pandas as pd
from datetime import datetime

#stock tickers
tickers = {
    'JPM': 'Finance - JPMorgan Chase',
    'TSLA': 'Technology - Tesla',
    'TGT': 'Retail - Target'
}

# Time range
start_date = '2010-01-01'
end_date = '2022-12-31'

# Download stock data
print("Collecting stock price data...\n")
stock_data = {}

for ticker, description in tickers.items():
    print(f"Downloading {description} ({ticker})...")
    stock_data[ticker] = yf.download(ticker, start=start_date, end=end_date, progress=False)
    print(f"  → Collected {len(stock_data[ticker])} observations")
    
print("\n" + "="*60)
print("Sample of JPMorgan Chase (JPM) data:")
print("="*60)
display(stock_data['JPM'].head(10))

print("\n" + "="*60)
print("Available numeric features:")
print("="*60)
print(stock_data['JPM'].columns.tolist())


  → Collected 3272 observations
  → Collected 3150 observations
  → Collected 3272 observations

Sample of JPMorgan Chase (JPM) data:


C:\Users\jannu\AppData\Local\Temp\ipykernel_10756\2801317572.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data[ticker] = yf.download(ticker, start=start_date, end=end_date, progress=False)
C:\Users\jannu\AppData\Local\Temp\ipykernel_10756\2801317572.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data[ticker] = yf.download(ticker, start=start_date, end=end_date, progress=False)
C:\Users\jannu\AppData\Local\Temp\ipykernel_10756\2801317572.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data[ticker] = yf.download(ticker, start=start_date, end=end_date, progress=False)


Price,Close,High,Low,Open,Volume
Ticker,JPM,JPM,JPM,JPM,JPM
Date,,,,,
2010-01-04,28.690948,28.784690,27.900859,27.981209,35460500
2010-01-05,29.246693,29.353823,28.644081,28.650778,41208300
2010-01-06,29.407398,29.521226,28.998964,29.092703,27729000
2010-01-07,29.989910,30.210866,29.199821,29.320343,44864700
2010-01-08,29.916262,29.929653,29.514522,29.708695,33110100
2010-01-11,29.815825,30.257739,29.668521,30.210869,31878700
2010-01-12,29.119471,29.561385,28.751207,29.454254,47109600
2010-01-13,29.628357,29.815835,28.704353,29.159659,39622000



Available numeric features:
[('Close', 'JPM'), ('High', 'JPM'), ('Low', 'JPM'), ('Open', 'JPM'), ('Volume', 'JPM')]


The stock data provides multiple numeric features including:
- Open: Opening price for the trading day
- High: Highest price during the trading day
- Low: Lowest price during the trading day  
- Close: Closing price for the trading day
- Adj Close: Adjusted closing price (accounts for dividends and splits)
- Volume: Number of shares traded

The ticker symbol itself provides a *categorical feature* (JPM, TSLA, or TGT), and dates can 
be transformed into additional categorical variables such as year, quarter, month, or pre/post
major economic events (e.g., pre-COVID vs. post-COVID periods).


#### Unconventional Variable Data

In order to investigate seemingly spurious correlations, we will also collect data from various unconventional sources
Below we access Google Trends data, which can track the popularity of certain search terms over time.


In [69]:
from pytrends.request import TrendReq
import time

# Initialize pytrends
pytrends = TrendReq(hl='en-US', tz=360)

print("Collecting Google Trends data for potentially correlated search terms...\n")

# Example searches that might correlate with our stocks
search_terms = {
    'robots': 'Tesla/Technology indicator',
    'back to school': 'Target/Retail indicator',
    'mortgage rates': 'JPMorgan/Finance indicator'
}

trends_data = {}

for term, description in search_terms.items():
    print(f"Downloading trends for '{term}' ({description})...")
    try:
        pytrends.build_payload([term], timeframe='2010-01-01 2022-12-31')
        trends_data[term] = pytrends.interest_over_time()
        if len(trends_data[term]) > 0:
            print(f"  → Collected {len(trends_data[term])} observations")
        time.sleep(1)  # Respect API rate limits
    except Exception as e:
        print(f"  → Error: {e}")

# Display sample
if 'robots' in trends_data and len(trends_data['robots']) > 0:
    print("\n" + "="*60)
    print("Sample Google Trends data for 'robots':")
    print("="*60)
    display(trends_data['robots'].head(10))


  → Collected 156 observations
  → Collected 156 observations
  → Collected 156 observations

Sample Google Trends data for 'robots':


,robots,isPartial
date,,
2010-01-01,38,False
2010-02-01,42,False
2010-03-01,40,False
2010-04-01,41,False
2010-05-01,41,False
2010-06-01,35,False
2010-07-01,33,False
2010-08-01,36,False
2010-09-01,36,False


Additional data sources to be incorporated in Phase II include:
- Demographic data: Student enrollment numbers from the National Center for Education Statistics (NCES)
- Weather data: Temperature and precipitation from NOAA
- Economic indicators: Unemployment rates, GDP growth from FRED API
- Cultural trends: Additional Google Trends data, social media metrics

All of these sources provide *numeric features* (counts, temperatures, percentages) that can be 
tested for correlation with stock prices. The variety of data sources allows for exploration of 
spurious correlations across multiple domains.


### Data Usage and Analysis Plan

The collected data will be used to address both research questions through machine learning techniques. 

For the first question—identifying which unusual variables correlate with stock prices—We will use regression 
analysis to measure the strength of relationships between stock prices and unconventional indicators like 
Google search trends and demographic data. This will help determine whether these strange variables have 
any predictive power. For the second question—understanding why these correlations exist—We will examine 
the data for confounding variables that might explain the relationships. For example, if multiple unusual 
variables all correlate with stock prices during the same time periods, this might indicate they are all 
responding to broader economic conditions like recessions or market booms rather than having direct connections. 
The analysis will help distinguish between correlations that reflect genuine economic relationships and those 
that are purely coincidental. This has practical importance because investors who mistake coincidental patterns 
for real indicators could make poor financial decisions based on misleading data.

### References

Vigen, Tyler. *Spurious Correlations*. Hachette Books, 2015.